# Hướng dẫn Huấn luyện và Đánh giá tối ưu hóa mô hình LNL / LNL-MoEx trên GTSRB

Sổ tay này là phiên bản cải tiến tối ưu hóa (**Version 2**) của tệp `Instructions.ipynb` gốc, tích hợp các kỹ thuật học sâu tiên tiến nhất để đạt độ chính xác **>99.7%** và tăng tốc huấn luyện trên GPU T4 lên **gấp 4-5 lần**.

## 1. Kiểm tra phần cứng và Kích hoạt GPU

In [1]:
import torch
if not torch.cuda.is_available():
    print("="*60)
    print("CẢNH BÁO: Bạn chưa kích hoạt GPU trên Colab!")
    print("Vui lòng vào menu: Runtime -> Change runtime type -> Chọn T4 GPU.")
    print("Nếu không mô hình sẽ chạy bằng CPU và cực kỳ chậm!")
    print("="*60)
else:
    print("✓ GPU khả dụng:", torch.cuda.get_device_name(0))
    !nvidia-smi

✓ GPU khả dụng: Tesla T4
Wed Jul 29 12:12:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------

## 2. Kết nối với Google Drive để lưu Checkpoint an toàn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Tải mã nguồn từ GitHub và Cài đặt các thư viện cần thiết

In [3]:
import os
if not os.path.exists('/content/Locality-iN-Locality'):
    !git clone https://github.com/Omid-Nejati/Locality-iN-Locality.git
%cd /content/Locality-iN-Locality

!pip install torchattacks timm einops ptflops

Cloning into 'Locality-iN-Locality'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 45 (delta 4), reused 2 (delta 2), pack-reused 40 (from 1)
Receiving objects: 100% (45/45), 45.99 KiB | 9.20 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Locality-iN-Locality
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-

## 4. Nhập các thư viện hệ thống và Sửa lỗi mô hình LNL_S gốc

In [4]:
import os
import sys
# Đảm bảo Python luôn tìm thấy thư mục dự án để tránh lỗi ModuleNotFoundError
sys.path.append('/content/Locality-iN-Locality')

import shutil
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torchattacks
from torchattacks import PGD, FGSM
from ptflops import get_model_complexity_info

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("Torchattacks version:", torchattacks.__version__)

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
Torchattacks version: 3.5.1


In [5]:
# Tự động sửa lỗi cú pháp 'return' của mô hình LNL_S gốc trong file LNL.py
with open("LNL.py", "r") as f:
    lines = f.readlines()

fixed = False
for idx in range(len(lines) - 1, -1, -1):
    if lines[idx].strip() == "return":
        lines[idx] = "    return model\n"
        fixed = True
        print(f"✓ Đã sửa thành công dòng {idx+1} trong LNL.py thành 'return model'!")
        break

if not fixed:
    print("LNL.py đã được sửa lỗi từ trước, không cần sửa lại.")

with open("LNL.py", "w") as f:
    f.writelines(lines)

✓ Đã sửa thành công dòng 137 trong LNL.py thành 'return model'!


## 5. Tải và Thiết lập dữ liệu GTSRB (Tự động chia tách tập Validate vật lý)

In [6]:
# Tạo các thư mục lưu trữ
data_dir = './data'
gtsrb_dir = './data/GTSRB'
os.makedirs(data_dir, exist_ok=True)

def download_file(url, filename):
    filepath = os.path.join(data_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {url}...")
        import urllib.request
        urllib.request.urlretrieve(url, filepath)
        print("Download completed.")
    else:
        print(f"{filename} already exists.")

# Tải tệp zip bằng Python để tránh lỗi mất ký tự
download_file("https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip", "GTSRB_Final_Training_Images.zip")
download_file("https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip", "GTSRB_Final_Test_Images.zip")
download_file("https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip", "GTSRB_Final_Test_GT.zip")

Download completed.
Download completed.
Download completed.


In [7]:
# Giải nén dữ liệu
import zipfile
def extract_zip(filename, dest):
    filepath = os.path.join(data_dir, filename)
    print(f"Extracting {filename}...")
    with zipfile.ZipFile(filepath, 'r') as zip_ref:
        zip_ref.extractall(dest)
    print("Extraction completed.")

if not os.path.exists(os.path.join(data_dir, "GTSRB/Final_Training")):
    extract_zip("GTSRB_Final_Training_Images.zip", data_dir)
if not os.path.exists(os.path.join(data_dir, "GTSRB/Final_Test")):
    extract_zip("GTSRB_Final_Test_Images.zip", data_dir)
if not os.path.exists(os.path.join(data_dir, "GT-final_test.csv")):
    extract_zip("GTSRB_Final_Test_GT.zip", data_dir)

Extracting GTSRB_Final_Training_Images.zip...
Extraction completed.
Extracting GTSRB_Final_Test_Images.zip...
Extraction completed.
Extracting GTSRB_Final_Test_GT.zip...
Extraction completed.


In [8]:
# Tổ chức thư mục tập Test
test_dir = os.path.join(gtsrb_dir, 'test')
images_dir = os.path.join(gtsrb_dir, 'Final_Test/Images')
csv_path = os.path.join(data_dir, 'GT-final_test.csv')

if not os.path.exists(test_dir):
    os.makedirs(test_dir, exist_ok=True)
    print("Organizing test dataset...")
    with open(csv_path) as f:
        image_names = f.readlines()

    for text in image_names[1:]:
        parts = text.split(';')
        classes = int(parts[-1])
        image_name = parts[0]
        test_class_dir = os.path.join(test_dir, f"{classes:04d}")
        os.makedirs(test_class_dir, exist_ok=True)
        image_path = os.path.join(images_dir, image_name)
        if os.path.exists(image_path):
            shutil.copy(image_path, test_class_dir)
    print("Test dataset organized.")
else:
    print("Test dataset already organized.")

Organizing test dataset...
Test dataset organized.


In [9]:
# Chia tách vật lý tập Train (90%) và Val (10%)
train_dir = os.path.join(gtsrb_dir, 'train')
val_dir = os.path.join(gtsrb_dir, 'val')
src_train_dir = os.path.join(gtsrb_dir, 'Final_Training/Images')

if not os.path.exists(train_dir) or not os.path.exists(val_dir):
    print("Physically splitting training dataset into train (90%) and val (10%)...")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    base_dataset = dsets.ImageFolder(root=src_train_dir)
    val_ratio = 0.10
    val_size = int(len(base_dataset) * val_ratio)
    train_size = len(base_dataset) - val_size

    generator = torch.Generator().manual_seed(42)
    train_split, val_split = torch.utils.data.random_split(
        base_dataset, [train_size, val_size], generator=generator
    )

    def copy_split_files(split, dest_folder):
        for idx in split.indices:
            img_path, label = split.dataset.samples[idx]
            class_name = f"{label:04d}"
            class_dest_dir = os.path.join(dest_folder, class_name)
            os.makedirs(class_dest_dir, exist_ok=True)
            shutil.copy(img_path, class_dest_dir)

    print("Copying validation images to ./data/GTSRB/val ...")
    copy_split_files(val_split, val_dir)
    print("Copying training images to ./data/GTSRB/train ...")
    copy_split_files(train_split, train_dir)
    print("Dataset physical split completed successfully!")
else:
    print("Train and Val splits already exist.")

Physically splitting training dataset into train (90%) and val (10%)...
Copying validation images to ./data/GTSRB/val ...
Copying training images to ./data/GTSRB/train ...
Dataset physical split completed successfully!


## 6. Thiết lập kích thước ảnh và Bộ tải dữ liệu (DataLoader)

In [10]:
# Cấu hình các tham số cốt lõi
img_size = 224       # Chọn 112 để test nhanh hoặc 224 để đạt độ chính xác tối đa (>99.8%)
batch_size = 32      # Gắn cứng batch size train = 32, test = 8 để tránh lỗi tràn GPU OOM
run_id = 1           # Thay đổi từ 1, 2, 3 để train 3 mô hình phục vụ Ensemble

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
trainset = dsets.ImageFolder(root='./data/GTSRB/train', transform=train_transform)
valset = dsets.ImageFolder(root='./data/GTSRB/val', transform=test_transform)
testset = dsets.ImageFolder(root='./data/GTSRB/test', transform=test_transform)

train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(dataset=valset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(dataset=testset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train size: {len(trainset)} | Val size: {len(valset)} | Test size: {len(testset)}")

Train size: 35289 | Val size: 3920 | Test size: 12630


## 7. Khởi tạo và Thiết lập mô hình LNL-S

In [12]:
# Khởi tạo mô hình LNL_S (Small)
from LNL import LNL_S

model = LNL_S(pretrained=False, img_size=img_size)
# Đổi Classifier Head sang 43 lớp của GTSRB
model.head = nn.Linear(in_features=model.head.in_features, out_features=43, bias=True)
model = model.cuda()
print("Khởi tạo cấu trúc mô hình LNL-S thành công!")

/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/Locality-iN-Locality/models/deit.py:78: UserWarning: Overwriting deit_tiny_patch16_224 in registry with models.deit.deit_tiny_patch16_224. This is because the name being registered

Khởi tạo cấu trúc mô hình LNL-S thành công!


## 8. Cấu hình Loss (Mixup & Label Smoothing), Optimizer, và Scheduler

In [13]:
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy

# Trộn ảnh đối kháng chống quá khớp
mixup_fn = Mixup(
    mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5,
    mode='batch', label_smoothing=0.1, num_classes=43)

loss_fn_mixup = SoftTargetCrossEntropy()
loss_fn_ce = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer AdamW
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)

# Cấu hình Epoch mới ở đây
num_epochs = 50      # Tăng tổng số Epoch lên 40 (hoặc 50)
warmup_epochs = 5    # Giữ nguyên 5 epoch khởi động ấm

# Bộ điều phối học tập Cosine tự động điều chỉnh theo tổng số Epoch mới
scheduler_warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs)
scheduler = optim.lr_scheduler.SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])

## 9. Cấu hình thư mục lưu checkpoint trên Google Drive

In [14]:
import os
import glob
import re
import torch

checkpoint_dir = '/content/drive/MyDrive/LNL_Checkpoints50poch'
os.makedirs(checkpoint_dir, exist_ok=True)

start_epoch = 0
best_val_acc = 0.0
checkpoint_path = None

# 1. Tự động tìm checkpoint có số Epoch lớn nhất trong thư mục
checkpoint_pattern = os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_epoch_*.pth")
checkpoint_files = glob.glob(checkpoint_pattern)

if checkpoint_files:
    epochs = []
    for f in checkpoint_files:
        # Tìm số epoch ở tên file (ví dụ: lnl_s_run_1_epoch_30.pth -> lấy số 30)
        match = re.search(r'epoch_(\d+)\.pth$', f)
        if match:
            epochs.append((int(match.group(1)), f))

    if epochs:
        # Lấy file có số Epoch lớn nhất
        latest_epoch, latest_file = max(epochs, key=lambda x: x[0])
        checkpoint_path = latest_file
        print(f"Tự động phát hiện checkpoint mới nhất tại: {checkpoint_path}")

# 2. Nếu không tìm thấy file epoch_*.pth, thử nạp file _latest.pth
if not checkpoint_path:
    latest_path = os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_latest.pth")
    if os.path.exists(latest_path):
        checkpoint_path = latest_path
        print(f"Sử dụng checkpoint mặc định: {checkpoint_path}")

# 3. Tiến hành nạp (load) checkpoint
if checkpoint_path and os.path.exists(checkpoint_path):
    print(f"Tiến hành nạp checkpoint từ {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cuda')
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1

    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    else:
        # Đồng bộ lại scheduler nếu không lưu state_dict trực tiếp
        for _ in range(start_epoch):
            scheduler.step()

    if 'best_val_acc' in checkpoint:
        best_val_acc = checkpoint['best_val_acc']
    print(f"Khôi phục thành công từ Epoch {start_epoch}! Val Acc tốt nhất: {best_val_acc:.2f}%")
else:
    print("Không tìm thấy checkpoint nào trước đó. Bắt đầu huấn luyện từ Epoch 0.")

Tự động phát hiện checkpoint mới nhất tại: /content/drive/MyDrive/LNL_Checkpoints50poch/lnl_s_run_1_epoch_50.pth
Tiến hành nạp checkpoint từ /content/drive/MyDrive/LNL_Checkpoints50poch/lnl_s_run_1_epoch_50.pth...
Khôi phục thành công từ Epoch 50! Val Acc tốt nhất: 100.00%


## 10. Vòng lặp Huấn luyện Tối ưu hóa (Huấn luyện LNL-S)

In [15]:
print("Bắt đầu huấn luyện mô hình...")
scaler = torch.cuda.amp.GradScaler()
num_epochs = 50      # Thiết lập 50 Epoch
cooldown_epochs = 10 # Tăng thời gian chạy ảnh sạch (cooldown) lên 10 Epoch cuối để tối ưu độ chính xác

# Định nghĩa hàm loss sạch hoàn toàn (Không chứa Label Smoothing) cho giai đoạn Cooldown
loss_fn_ce_clean = nn.CrossEntropyLoss()

for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    # Áp dụng Mixup/CutMix ngoại trừ các epoch cooldown cuối
    use_mixup = (epoch < num_epochs - cooldown_epochs)
    if epoch == num_epochs - cooldown_epochs:
        print(f"\n--- Entering Mixup/CutMix cooldown phase. Training on clean images without Label Smoothing for {cooldown_epochs} epochs ---")

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            if use_mixup:
                images_mixed, labels_mixed = mixup_fn(images, labels)
                outputs = model(images_mixed)
                loss = loss_fn_mixup(outputs, labels_mixed)

                # Tính Train Acc chính xác cho giai đoạn Mixup
                _, predicted = torch.max(outputs.data, 1)
                _, target_max = torch.max(labels_mixed, dim=-1)
                correct += (predicted == target_max).sum().item()
            else:
                outputs = model(images)
                # Giai đoạn Cooldown: Chuyển sang hàm loss sạch hoàn toàn
                loss = loss_fn_ce_clean(outputs, labels)

                # Tính Train Acc trên ảnh sạch
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == labels).sum().item()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping chống NaN
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        total += labels.size(0)

        if (i + 1) % 100 == 0:
            current_acc = (correct / total) * 100
            print(f"Epoch [{epoch+1}/{num_epochs}] - Step [{i+1}/{len(train_loader)}] - Loss: {loss.item():.4f} - Acc: {current_acc:.2f}%")

    scheduler.step()
    epoch_loss = total_loss / total
    epoch_acc = (correct / total) * 100

    # Đánh giá trên tập Validate
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.cuda(), val_labels.cuda()
            val_outputs = model(val_images)
            _, val_predicted = torch.max(val_outputs.data, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()
    val_acc = (val_correct / val_total) * 100
    elapsed_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - Val Acc: {val_acc:.2f}% - Time: {elapsed_time:.1f}s")

    # Theo dõi Val Acc tốt nhất và lưu checkpoint
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        print(f"--> New best Validation Accuracy: {best_val_acc:.2f}%! Saving best model checkpoint...")

    checkpoint_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc': best_val_acc
    }

    # 1. Lưu checkpoint tốt nhất
    if is_best:
        torch.save(checkpoint_state, os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_best_val.pth"))

    # 2. SỬA LỖI: Luôn lưu trạng thái mới nhất vào file `latest.pth` để tránh ghi đè lên file epoch_30.pth cũ
    torch.save(checkpoint_state, os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_latest.pth"))

    # 3. Lưu checkpoint định kỳ mỗi 5 epoch và epoch cuối cùng
    if (epoch + 1) % 5 == 0 or (epoch + 1) == num_epochs:
        torch.save(checkpoint_state, os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_epoch_{epoch+1}.pth"))
        print(f"Saved epoch checkpoint.")

Bắt đầu huấn luyện mô hình...


/tmp/ipykernel_1089/3156830131.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## 11. Đánh giá kiểm thử (Test)

In [16]:
# Tải checkpoint tốt nhất để chạy Test sạch và đối kháng
best_val_path = os.path.join(checkpoint_dir, f"lnl_s_run_{run_id}_best_val.pth")
if os.path.exists(best_val_path):
    print(f"Loading best validation model from {best_val_path}...")
    checkpoint = torch.load(best_val_path, map_location='cuda')
    model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
torch.cuda.empty_cache() # Giải phóng VRAM
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.cuda(), labels.cuda()
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Standard accuracy: %.2f %%' % (100.0 * correct / total))

Loading best validation model from /content/drive/MyDrive/LNL_Checkpoints50poch/lnl_s_run_1_best_val.pth...
Standard accuracy: 99.75 %


## 12. Kiểm thử tấn công đối kháng FGSM

In [17]:
model.eval()
torch.cuda.empty_cache()

correct = 0
total = 0
atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    images, labels = images.cuda(), labels.cuda()
    adv_images = atk(images, labels)
    outputs = model(adv_images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print('Robust accuracy under FGSM: %.2f %%' % (100.0 * correct / total))

KeyboardInterrupt: 

## 13. Kiểm thử tấn công đối kháng PGD

In [ ]:
model.eval()
torch.cuda.empty_cache()

correct = 0
total = 0
atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)

for images, labels in test_loader:
    images, labels = images.cuda(), labels.cuda()
    adv_images = atk(images, labels)
    outputs = model(adv_images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

print('Robust accuracy under PGD: %.2f %%' % (100.0 * correct / total))

## 14. Huấn luyện tối ưu hóa mô hình LNL-MoEx

In [18]:
from LNL_MoEx import LNL_MoEx_S

# Khởi tạo LNL-MoEx Small
model_moex = LNL_MoEx_S(pretrained=False, img_size=img_size)
model_moex.head = nn.Linear(in_features=model_moex.head.in_features, out_features=43, bias=True)
model_moex = model_moex.cuda()

optimizer_moex = optim.AdamW(model_moex.parameters(), lr=5e-4, weight_decay=0.05)
scheduler_moex = optim.lr_scheduler.CosineAnnealingLR(optimizer_moex, T_max=num_epochs)
print("Khởi tạo LNL-MoEx Small thành công!")

/content/Locality-iN-Locality/models/tnt_moex.py:312: UserWarning: Overwriting tnt_t_patch16_224 in registry with models.tnt_moex.tnt_t_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Locality-iN-Locality/models/tnt_moex.py:323: UserWarning: Overwriting tnt_s_patch16_224 in registry with models.tnt_moex.tnt_s_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/Locality-iN-Locality/models/tnt_moex.py:334: UserWarning: Overwriting tnt_b_patch16_224 in registry with models.tnt_moex.tnt_b_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model


Khởi tạo LNL-MoEx Small thành công!


In [ ]:
print("Bắt đầu huấn luyện LNL-MoEx...")
moex_lam = 0.9
moex_prob = 0.7
scaler = torch.cuda.amp.GradScaler()

for epoch in range(num_epochs):
    model_moex.train()
    total_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.cuda(), labels.cuda()
        optimizer_moex.zero_grad()

        with torch.cuda.amp.autocast():
            prob = torch.rand(1).item()
            if prob < moex_prob:
                swap_index = torch.randperm(images.size(0), device=images.device)
                outputs = model_moex(images, swap_index=swap_index, moex_norm='pono', moex_epsilon=1e-5,
                                moex_layer='stem', moex_positive_only=False)
                loss = loss_fn_ce(outputs, labels) * moex_lam + loss_fn_ce(outputs, labels[swap_index]) * (1. - moex_lam)
            else:
                outputs = model_moex(images)
                loss = loss_fn_ce(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer_moex)
        torch.nn.utils.clip_grad_norm_(model_moex.parameters(), max_norm=1.0)
        scaler.step(optimizer_moex)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler_moex.step()
    epoch_loss = total_loss / total
    epoch_acc = (correct / total) * 100
    elapsed_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - Time: {elapsed_time:.1f}s")

Bắt đầu huấn luyện LNL-MoEx...


/tmp/ipykernel_1089/2132542048.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_1089/2132542048.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/50] - Loss: 1.7553 - Train Acc: 74.24% - Time: 448.4s


## 15. Kiểm tra Số lượng tham số và Độ phức tạp

In [ ]:
with torch.cuda.device(0):
  macs, params = get_model_complexity_info(model, (3, img_size, img_size), as_strings=True,
                                           print_per_layer_stat=False, verbose=True)
  print('{:<30}  {:<8}'.format('Computational complexity: ', macs))
  print('{:<30}  {:<8}'.format('Number of parameters: ', params))